In [19]:
import os
import pickle
import pandas as pd
import glob
from tqdm import tqdm, trange

# math imports
import numpy as np
import scipy
import sklearn

from dataclasses import dataclass

import matplotlib
from matplotlib import pyplot

import copy
import sys
sys.path.append('../../../')
from src.data_class import matrix_class

In [20]:
def has_numbers(inputString):
    return any(char.isdigit() for char in inputString)

import warnings
warnings.simplefilter("ignore")

In [21]:
HBN_basic_demos = pd.read_csv("../9994_Basic_Demos_20210310.csv", low_memory=False)
HBN_basic_demos = HBN_basic_demos.iloc[1:,:]
HBN_basic_demos[["Age", "Sex"]] = HBN_basic_demos[["Age", "Sex"]].apply(pd.to_numeric)
min_age = np.min( HBN_basic_demos['Age'].values )
max_age = np.max( HBN_basic_demos['Age'].values )

metadata = ['EID', 'Sex', 'Age']

print('Max age {}, min age {}'.format(max_age, min_age))
print('Sex code : Male = 0, Female = 1')

Max age 21.899041, min age 5.003878
Sex code : Male = 0, Female = 1


In [22]:
dataname = 'CBCL'
filename = '../9994_{}_20210310.csv'.format(dataname)
dict_filename = '../extra_questionnaire/dictionary/{}.xlsx'.format(dataname)

data = pd.read_csv(filename, low_memory=False)
data = data.iloc[1:,:]

ID_index = list(data.columns).index('Anonymized ID')
EID_index = list(data.columns).index('EID')

In [23]:
duplicates = data.loc[data['EID'].isin( data.loc[data.duplicated(subset=['EID'])]['EID'] )]
duplicates = duplicates.sort_values(by='EID')

In [24]:
response_index = [(dataname in col) and
                  (has_numbers(col.replace(dataname+'_', ''))) and
                  ('113' not in col)
                  for col in data.columns]
response_index = list(np.where(response_index)[0])
response_abbre = [data.columns[idx] for idx in response_index]
# print(','.join(response_abbre))

dictionary = pd.read_excel(dict_filename, header=1)
response_info = [dictionary.loc[dictionary['Variable']==v]['Question'].values.squeeze() for v in response_abbre]
response = list(zip(response_abbre, response_info))
assert len(response) > 0
print('\n'.join([r[0] + ' : ' + r[1] for r in response]))

data[response_abbre] = data[response_abbre].apply(pd.to_numeric)
df = data.copy()
df = df.iloc[:, [EID_index]+response_index]
df = df.merge(HBN_basic_demos[metadata], 'left')
df[['Sex', 'Age']] = df[['Sex', 'Age']].apply(pd.to_numeric)
df.insert(0, 'Sex', df.pop('Sex'))
df.insert(0, 'Age', df.pop('Age'))

# ===
df = df.drop_duplicates(subset=['Age', 'EID']+response_abbre)
df = df.drop_duplicates(subset=['EID'], keep='last')
# ===

assert np.sum(np.isnan(df[['Sex', 'Age']].values)) == 0

print('Unique subjects : {}'.format(np.unique(df['EID'].values).shape[0]))
print('Data size : {} x {}'.format(df.shape[0], df.shape[1]))

CBCL_01 : 1. Acts too young for his/her age
CBCL_02 : 2. Drinks alcohol without parents' approval
CBCL_03 : 3. Argues a lot
CBCL_04 : 4. Fails to finish things he/she starts
CBCL_05 : 5. There is very little he/she enjoys
CBCL_06 : 6. Bowel movements outside toilet
CBCL_07 : 7. Bragging, boasting
CBCL_08 : 8. Can't concentrate, can't pay attention for long
CBCL_09 : 9. Can't get his/her mind off certain thoughts; obsessions
CBCL_10 : 10. Can't sit still, restless or hyperactive
CBCL_11 : 11. Clings to adults or too dependent
CBCL_12 : 12. Complains of loneliness
CBCL_13 : 13. Confused or seems to be in a fog
CBCL_14 : 14. Cries a lot
CBCL_15 : 15. Cruel to animals
CBCL_16 : 16. Cruelty, bullying, or meanness to others
CBCL_17 : 17. Daydreams or gets lost in his/her thoughts
CBCL_18 : 18. Deliberately harms self or attempts suicide
CBCL_19 : 19. Demands a lot of attention
CBCL_20 : 20. Destroys his/her own things
CBCL_21 : 21. Destroys things belonging to his/her family or others
CBCL_2

In [26]:
subscale_abbre = ['CBCL_AD', 'CBCL_AD_T',
                  'CBCL_WD', 'CBCL_WD_T',
                  'CBCL_SC', 'CBCL_SC_T',
                  'CBCL_SP', 'CBCL_SP_T',
                  'CBCL_TP', 'CBCL_TP_T',
                  'CBCL_AP', 'CBCL_AP_T',
                  'CBCL_RBB', 'CBCL_RBB_T',
                  'CBCL_AB', 'CBCL_AB_T',
                  'CBCL_OP', 'CBCL_Int',
                  'CBCL_Int_T', 'CBCL_Ext',
                  'CBCL_Ext_T', 'CBCL_C',
                  'CBCL_Total', 'CBCL_TOTAL_T'
                 ]
subscale_index = [list(data.columns).index(s) for s in subscale_abbre]
subscale_info = [dictionary.loc[dictionary['Variable']==v]['Question'].values.squeeze() for v in subscale_abbre]
subscale_info = [str(s) for s in subscale_info]
subscale_info[-1] = 'Total T Score'

subscale = list(zip(subscale_abbre, subscale_info))

print('\n'.join([s[0] + ' : ' + s[1] for s in subscale]))

data[subscale_abbre] = data[subscale_abbre].apply(pd.to_numeric)
df_subscale = data.copy()
df_subscale = df_subscale.iloc[:, [EID_index]+subscale_index]
df_subscale = df_subscale.merge(HBN_basic_demos[metadata], 'left')
df_subscale[['Sex', 'Age']] = df_subscale[['Sex', 'Age']].apply(pd.to_numeric)
df_subscale.insert(0, 'Sex', df_subscale.pop('Sex'))
df_subscale.insert(0, 'Age', df_subscale.pop('Age'))

# ===
df_subscale = df_subscale.drop_duplicates(subset=['Age', 'EID']+subscale_abbre)
df_subscale = df_subscale.drop_duplicates(subset=['EID'], keep='last')
# ===

assert np.sum(np.isnan(df_subscale[['Sex', 'Age']].values)) == 0

print('Unique subjects : {}'.format(np.unique(df_subscale['EID'].values).shape[0]))
print('Data size : {} x {}'.format(df_subscale.shape[0], df_subscale.shape[1]))


CBCL_AD : Anxious/Depressed Raw Score
CBCL_AD_T : Anxious/Depressed T Score
CBCL_WD : Withdrawn/Depressed Raw Score
CBCL_WD_T : Withdrawn/Depressed T Score
CBCL_SC : Somatic Complaints Raw Score
CBCL_SC_T : Somatic Complaints T Score
CBCL_SP : Social Problems Raw Score
CBCL_SP_T : Social Problems T Score
CBCL_TP : Thought Problems Raw Score
CBCL_TP_T : Thought Problems T Score
CBCL_AP : Attention Problems Raw Score
CBCL_AP_T : Attention Problems T Score
CBCL_RBB : Rule Breaking Behavior Raw Score
CBCL_RBB_T : Rule Breaking Behavior T Score
CBCL_AB : Aggressive Behavior Raw Score
CBCL_AB_T : Aggressive Behavior T Score
CBCL_OP : Other Problems Raw Score
CBCL_Int : Internalizing Raw Score
CBCL_Int_T : Internalizing T Score
CBCL_Ext : Externalizing Raw Score
CBCL_Ext_T : Externalizing T Score
CBCL_C : C Score Raw Score
CBCL_Total : Total Raw Score
CBCL_TOTAL_T : Total T Score
Unique subjects : 3094
Data size : 3094 x 27


In [27]:
max_response = 2.0
min_response = 0.0

max_subscale = np.nanmax(df_subscale[subscale_abbre].values, axis=0)
min_subscale = np.nanmin(df_subscale[subscale_abbre].values, axis=0)


### Load CBCL questions information (2001 version)
- The questions are the 2001 version (same for both ABCD and HBN)
- The third column in the original csv file: `CBCL2001_item_variables.csv` corresponds to the grouping of questions
- Typo fixed for item 56e and a new csv file `CBCL2001_item_variables_fixed.csv` was created
- Introduce a new column to encode questions' group ID. For instance, 4-002 stands for Question Group 4, second question

In [28]:
CBCL_question = pd.read_csv('../CBCL2001_item_variables_fixed.csv', index_col=False)
CBCL_question = CBCL_question.iloc[:-1,:]
QuestionClass = list(np.unique(CBCL_question['CBCL2001_6-18_scale'].values))
group_index = []
for k in range(CBCL_question.shape[0]):
    qid = CBCL_question.loc[k]['CBCL2001_6-18_varname'].split('.')[0]
    group_index.append(str(QuestionClass.index(CBCL_question.loc[k]['CBCL2001_6-18_scale'])) + '-' + qid)
CBCL_question['group'] = group_index
question = CBCL_question['CBCL2001_6-18_varname'].values

In [17]:
os.makedirs('../processed/pickle', exist_ok=True)
os.makedirs('../processed/dataclass', exist_ok=True)
os.makedirs('../processed/npz', exist_ok=True)

with open(os.path.join('../processed/pickle', '{}.pickle'.format(dataname)), 'wb') as handle:
    pickle.dump({'data':df,
                 'data_info':'Child Behavior Checklist (CBCL) -- Age 6-18',
                 'response_info':response,
                 'CBCL_question': CBCL_question,
                 'data_subscale':df_subscale,
                 'subscale_info':subscale,
                 'max_response':max_response,
                 'min_response':min_response,
                 'max_subscale':max_subscale,
                 'min_subscale':min_subscale,
                 'max_age':max_age,
                 'min_age':min_age
                },
                handle,
                protocol=4)
    
itemlist = [r[0] for r in response]
subjlist = list(df['EID'].values)
    
M_raw = df[itemlist].values
nan_mask = 1.0 - np.isnan(M_raw)

M = copy.deepcopy(M_raw)
M[np.isnan(M)] = min_response
M = (M - min_response)/(max_response - min_response)
assert np.sum(np.isnan(M)) == 0

confound_raw = df[['Age', 'Sex']].values
confound = copy.deepcopy(confound_raw)
confound[:,0] = (confound[:,0] - min_age)/(max_age - min_age)
assert np.sum(np.isnan(confound)) == 0

matrix = matrix_class(M, M_raw,
                      confound, confound_raw,
                      nan_mask,
                      None, None, None, # row_idx, col_idx, mask
                      dataname,
                      subjlist,
                      response,
                      None, None, # W, Q
                      None, None, # C, Qc
                      None, None) # Z, aZ

with open(os.path.join('../processed/dataclass', '{}.pickle'.format(dataname)), 'wb') as handle:
    pickle.dump(matrix, handle, protocol=4)
    
np.savez(os.path.join('../processed/npz','{}.npz'.format(dataname)),
         M = matrix.M,
         nan_mask = matrix.nan_mask,
         cofounder = matrix.confound)